# Pandas Mini Proje: Perakende Satış Analizi

Bu notebook, Pandas konularını uçtan uca küçük bir veri bilimi senaryosunda birleştirir.

Yapılacaklar:

* Ham satış verisini oluşturma
* Veri temizleme
* Yeni sütunlar üretme
* Kategori ve şehir bazlı analiz
* Aylık trend çıkarma
* Müşteri segmenti üretme


## Konu Dokümantasyonu

Mini proje notebook'ları, Pandas konularını gerçekçi bir analiz akışı içinde birleştirir. Buradaki amaç tek tek komut ezberlemek değil, ham veriden raporlanabilir çıktıya giden yolu anlamaktır.

Bu Pandas mini projesinde perakende satış verisi üzerinde çalışılır. Veri içinde tekrar eden kayıt, eksik fiyat, metin tutarsızlığı ve tarih alanı bulunur. Bu yapı gerçek veri bilimi çalışmalarında sık karşılaşılan problemleri küçük ölçekte gösterir.

Bu notebook'ta kullanılan temel Pandas becerileri:

* Tekrarlı kayıtları temizleme
* Eksik değer doldurma
* Metin standardizasyonu
* Tarih dönüşümü
* Yeni sütun üretme
* `groupby()` ile özet çıkarma
* Müşteri segmentasyonu


## Senaryo

Bir perakende işletmesinin sipariş verisi var. Veri içinde eksik değer, tekrar eden sipariş ve metin tutarsızlıkları bulunuyor.

Gerçek projelerde veri nadiren temiz gelir. Bu yüzden önce temizleme, sonra analiz yapılmalıdır.


In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "siparis_id": [1, 2, 2, 3, 4, 5, 6],
    "tarih": ["2026-01-05", "2026-01-08", "2026-01-08", "2026-02-03", "2026-02-15", "2026-02-20", "2026-03-01"],
    "musteri": ["Ali", "Ayşe", "Ayşe", "Mehmet", "Zeynep", "Ali", "Can"],
    "sehir": [" İstanbul ", "ANKARA", "ANKARA", "İzmir", "istanbul", "İstanbul", "Ankara"],
    "kategori": ["Elektronik", "Mobilya", "Mobilya", "Elektronik", "Aksesuar", "Elektronik", "Mobilya"],
    "adet": [1, 2, 2, 1, 3, 1, 2],
    "birim_fiyat": [30000, 7500, 7500, 20000, 2500, np.nan, 9000],
})

print(df)


## Kolay Seviye: Veri Temizleme

Tekrar eden satırları kaldıralım, tarih sütununu dönüştürelim, şehir adlarını standartlaştıralım ve eksik fiyatı kategori medyanı ile dolduralım.


In [ ]:
import pandas as pd

df = df.drop_duplicates()
df["tarih"] = pd.to_datetime(df["tarih"])
df["sehir"] = df["sehir"].str.strip().str.lower()

kategori_medyan = df.groupby("kategori")["birim_fiyat"].transform("median")
df["birim_fiyat"] = df["birim_fiyat"].fillna(kategori_medyan)

print(df)
print(df.isna().sum())


## Orta Seviye: Yeni Değişkenler Üretme

Analizde kullanılacak yeni sütunlar üretelim:

* `ciro`: adet x birim fiyat
* `yil_ay`: aylık analiz için yıl-ay bilgisi
* `siparis_buyuklugu`: siparişi küçük/orta/büyük olarak etiketleme


In [ ]:
df["ciro"] = df["adet"] * df["birim_fiyat"]
df["yil_ay"] = df["tarih"].dt.to_period("M")

df["siparis_buyuklugu"] = pd.cut(
    df["ciro"],
    bins=[0, 5000, 20000, float("inf")],
    labels=["küçük", "orta", "büyük"],
)

print(df[["siparis_id", "kategori", "ciro", "siparis_buyuklugu"]])


## Orta Seviye: Kategori ve Şehir Analizi

GroupBy ile kategori ve şehir bazında toplam ciro hesaplayalım.


In [ ]:
kategori_ozeti = (
    df.groupby("kategori")
    .agg(
        toplam_ciro=("ciro", "sum"),
        ortalama_ciro=("ciro", "mean"),
        siparis_sayisi=("siparis_id", "count"),
    )
    .sort_values("toplam_ciro", ascending=False)
)

sehir_ozeti = df.groupby("sehir")["ciro"].sum().sort_values(ascending=False)

print("Kategori özeti:")
print(kategori_ozeti)
print("Şehir özeti:")
print(sehir_ozeti)


## İleri Seviye: Aylık Trend ve Müşteri Segmenti

Zaman kırılımı ve müşteri bazlı toplam harcama, iş analitiğinde sık kullanılan özetlerdir.


In [ ]:
aylik_trend = df.groupby("yil_ay")["ciro"].sum()

musteri_ozeti = (
    df.groupby("musteri")
    .agg(
        toplam_harcama=("ciro", "sum"),
        siparis_sayisi=("siparis_id", "count"),
    )
    .sort_values("toplam_harcama", ascending=False)
)

musteri_ozeti["segment"] = pd.cut(
    musteri_ozeti["toplam_harcama"],
    bins=[0, 10000, 30000, float("inf")],
    labels=["standart", "değerli", "premium"],
)

print("Aylık trend:")
print(aylik_trend)
print("Müşteri özeti:")
print(musteri_ozeti)


## Mini Sonuç

Bu mini projede Pandas ile uçtan uca veri manipülasyonu yaptık.

Ana adımlar:

* Veriyi tanıma
* Tekrarları temizleme
* Eksik değerleri doldurma
* Metin ve tarih sütunlarını düzenleme
* Yeni özellik üretme
* GroupBy ile özet çıkarma
* Müşteri segmenti oluşturma

Bu akış, gerçek veri bilimi projelerinde sık kullanılan temel bir çalışma düzenidir.


In [ ]:
final_rapor = {
    "toplam_ciro": df["ciro"].sum(),
    "toplam_siparis": df["siparis_id"].nunique(),
    "en_iyi_kategori": kategori_ozeti.index[0],
    "en_iyi_musteri": musteri_ozeti.index[0],
}

print(final_rapor)
